# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssof-Essam/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract (5 plain-words answers):**

1. **One row = one content item (page)**, aggregated over a prior 90-day feature window ending at the decision date.

2. **Tables**: `dim_content` (metadata) + `fact_content_daily_performance` (daily metrics, partitioned by `month=YYYY-MM`).

3. **Time window**: Iterate on mid-panel month **`month=2026-03`** as the outcome month. Feature window = 90 days before March 1 (Dec 2025–Feb 2026). Label = `trend_direction == "down"` in March 2026 (current-window proxy, consistent with w01/w02).

4. **Label/proxy**: `is_declining_label = (trend_direction == "down")` from the March 2026 slice.

5. **Excluded**: `trend_direction`, `trend_pct`, any target-window metrics (`*_last30`, `*_last7`) — label-derived, never features.

In [ ]:
# --- Token loader (Colab Secrets or .env) ---
import os
import sys

def get_hf_token():
    # 1. Try Colab secrets first
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    
    # 2. Fall back to .env file (local dev)
    from pathlib import Path
    env_path = Path(".env")
    if env_path.exists():
        from dotenv import load_dotenv
        load_dotenv()
        token = os.getenv("HF_TOKEN")
        if token:
            return token
    
    # 3. Fall back to env var (already set in shell)
    token = os.getenv("HF_TOKEN")
    if token:
        return token
    
    raise RuntimeError("HF_TOKEN not found. Set in Colab Secrets or .env file.")

HF_TOKEN = get_hf_token()
print("HF_TOKEN loaded")

# --- Load dimension tables via datasets library (handles gated access) ---
from datasets import load_dataset
import pandas as pd
import os

print("Loading dimension tables via datasets library...")
dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train",
    token=HF_TOKEN
).to_pandas()

dim_clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    token=HF_TOKEN
).to_pandas()

print(f"dim_content: {len(dim_content)} rows")
print(f"dim_clients: {len(dim_clients)} rows")

# Cache dimension tables locally
os.makedirs("work/outputs", exist_ok=True)
dim_content.to_parquet("work/outputs/dim_content.parquet", index=False)
dim_clients.to_parquet("work/outputs/dim_clients.parquet", index=False)
print("Cached dimension tables to work/outputs/")

# --- DuckDB for fact table (fast partition pruning) ---
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Cache directory
import os
os.makedirs("work/outputs", exist_ok=True)

# --- Cache March 2026 fact partition (fast with partition pruning) ---
cache_path = "work/outputs/month_2026_03.parquet"
if not os.path.exists(cache_path):
    print("Loading and caching March 2026 partition (partition pruning)...")
    query = f"SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
    df_march = con.execute(query).df()
    df_march.to_parquet(cache_path, index=False)
    print(f"Cached {len(df_march)} rows to {cache_path}")
else:
    df_march = pd.read_parquet(cache_path)
    print(f"Loaded cached March 2026: {len(df_march)} rows")

# Sample join (handle duplicate client_hash_id with suffixes)
sample = df_march.head(5).merge(dim_content, on="content_hash_id", how="left", suffixes=("", "_dim"))
# After merge: client_hash_id from fact, client_hash_id_dim from dim_content
show_cols = ["report_date", "client_hash_id", "content_hash_id",
             "gsc_impressions", "gsc_clicks", "gsc_avg_position",
             "ga4_sessions", "ga4_engagement_rate",
             "ga4_data_available", "gsc_data_available",
             "content_created_date", "days_since_last_update",
             "word_count", "content_type", "main_intent"]
# Only show columns that exist
available_cols = [c for c in show_cols if c in sample.columns]
print("Sample rows (month=2026-03, joined with dim_content):")
print(sample[available_cols].to_string(index=False))

## 2. Fields: feature / label / context / excluded

**Field classification for our Lane 2 slice:**

| Bucket | Columns | Why |
|--------|---------|-----|
| **Feature** | `log_impressions_90d`, `avg_position_90d` (excl 0), `ctr_90d`, `days_since_last_update`, `content_age_days`, `word_count` + `has_word_count`, `engagement_rate_90d`, `sessions_90d` | Aggregated from feature window (90 days before decision); knowable before we act |
| **Label/Proxy** | `is_declining_label` (`trend_direction == "down"` in March 2026) | Current-window proxy; what we predict |
| **Context** | `content_hash_id`, `client_hash_id`, `report_date` | Grouping, joining, splitting — never model features |
| **Excluded** | `trend_direction`, `trend_pct`, `gsc_impressions_last30`, `gsc_clicks_last30`, `ga4_sessions_last30`, any target-window metric | Label-derived / future information — leakage risk |

In [ ]:
# Show column -> bucket map for the joined frame
import pandas as pd

field_map = {
    # Features
    "log_impressions_90d": "feature",
    "avg_position_90d": "feature",
    "ctr_90d": "feature",
    "days_since_last_update": "feature",
    "content_age_days": "feature",
    "word_count": "feature",
    "has_word_count": "feature",
    "engagement_rate_90d": "feature",
    "sessions_90d": "feature",
    # Label / Proxy
    "is_declining_label": "label_proxy",
    # Context
    "content_hash_id": "context",
    "client_hash_id": "context",
    "report_date": "context",
    # Excluded (leakage)
    "trend_direction": "excluded",
    "trend_pct": "excluded",
    "gsc_impressions_last30": "excluded",
    "gsc_clicks_last30": "excluded",
    "ga4_sessions_last30": "excluded",
}

df_map = pd.DataFrame([{"column": k, "bucket": v} for k, v in field_map.items()])
print(df_map.to_string(index=False))

## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries on `month=2026-03`, each with output visible.

In [ ]:
# Query 1: Grain check — one row = one (report_date, client_hash_id, content_hash_id)
import pandas as pd

grain_check = df_march.groupby(["report_date", "client_hash_id", "content_hash_id"]).size().reset_index(name="count")
duplicates = grain_check[grain_check["count"] > 1]
print(f"Grain check — duplicate (date, client, content) rows: {len(duplicates)}")
if len(duplicates) == 0:
    print("OK: grain holds — one row per (date, client, content)")
else:
    print(duplicates.head())


In [ ]:
# Query 2: Counts + date span
total = len(df_march)
min_date = df_march["report_date"].min()
max_date = df_march["report_date"].max()
unique_contents = df_march["content_hash_id"].nunique()
unique_clients = df_march["client_hash_id"].nunique()

print("Counts + date span for month=2026-03:")
print(f"  total_rows: {total}")
print(f"  min_date: {min_date}")
print(f"  max_date: {max_date}")
print(f"  unique_contents: {unique_contents}")
print(f"  unique_clients: {unique_clients}")


In [ ]:
# Query 3: Availability filter with IS TRUE
total = len(df_march)
ga4_ok = int(df_march["ga4_data_available"].sum())
gsc_ok = int(df_march["gsc_data_available"].sum())
both_ok = int(((df_march["ga4_data_available"]) & (df_march["gsc_data_available"])).sum())

print("Availability filter (IS TRUE) for month=2026-03:")
print(f"  total_rows: {total}")
print(f"  ga4_available: {ga4_ok}")
print(f"  gsc_available: {gsc_ok}")
print(f"  both_available: {both_ok}")
print(f"\nSurvival rate with both flags IS TRUE: {both_ok}/{total} = {both_ok/total:.1%}")


## 4. Five features + "knowable at decision moment"

Feature frame built from the 90-day feature window (Dec 2025–Feb 2026) for content items present in March 2026.

| Feature | Knowable at decision moment because… |
|---------|--------------------------------------|
| `log_impressions_90d` | Sum of daily impressions over 90 days **before** decision date (Dec–Feb) |
| `avg_position_90d` | Mean `gsc_avg_position` (excluding 0 = no data) over prior 90 days |
| `ctr_90d` | `clicks_90d / impressions_90d * 100` from feature window only |
| `days_since_last_update` | From `dim_content` — static metadata, known at content creation |
| `content_age_days` | `(decision_date - content_created_date).days` — fully known at decision time |

In [ ]:
# Build 5-feature frame from 90-day feature window (Dec 2025 – Feb 2026)
# We aggregate daily fact for content items that appear in March 2026
import pandas as pd
import numpy as np
import os

# Load Dec 2025, Jan 2026, Feb 2026 partitions (feature window)
feature_months = ["2025-12", "2026-01", "2026-02"]
feature_dfs = []

for m in feature_months:
    cache_path = f"work/outputs/month_{m}.parquet"
    if not os.path.exists(cache_path):
        print(f"Loading and caching {m} partition (partition pruning)...")
        query = f"SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')"
        df = con.execute(query).df()
        df.to_parquet(cache_path, index=False)
        print(f"  Cached {len(df)} rows")
    else:
        df = pd.read_parquet(cache_path)
        print(f"  Loaded cached {m}: {len(df)} rows")
    feature_dfs.append(df)

df_feature = pd.concat(feature_dfs, ignore_index=True)
print(f"Total feature window rows: {len(df_feature)}")

# Filter to content IDs that appear in March 2026
march_ids = df_march["content_hash_id"].unique()
df_feature = df_feature[df_feature["content_hash_id"].isin(march_ids)].copy()
print(f"Feature window rows for March content: {len(df_feature)}")

# Filter: GSC available only
df_feature = df_feature[df_feature["gsc_data_available"] == True].copy()

# Aggregate to content level
agg = df_feature.groupby(["content_hash_id", "client_hash_id"]).agg(
    impressions_90d=("gsc_impressions", "sum"),
    clicks_90d=("gsc_clicks", "sum"),
    sessions_90d=("ga4_sessions", "sum"),
    avg_position_90d=("gsc_avg_position", lambda x: x[x > 0].mean() if (x > 0).any() else np.nan),
    engagement_rate_90d=("ga4_engagement_rate", "mean"),
).reset_index()

# Merge dim_content metadata (handle duplicate client_hash_id with suffixes)
feat_df = agg.merge(dim_content, on="content_hash_id", how="left", suffixes=("", "_dim"))
# After merge: client_hash_id from agg, client_hash_id_dim from dim_content

# Compute derived features
feat_df["log_impressions_90d"] = np.log1p(feat_df["impressions_90d"])
feat_df["ctr_90d"] = np.where(feat_df["impressions_90d"] > 0,
                                 feat_df["clicks_90d"] / feat_df["impressions_90d"] * 100, 0)
feat_df["has_word_count"] = feat_df["word_count"].notna().astype(int)
feat_df["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_created_date"])).dt.days

# Label placeholder (trend_direction not in warehouse)
feat_df["is_declining_label"] = 0

print(f"Aggregated feature rows: {len(feat_df)}")

print("Feature frame (first 10 rows):")
show_cols = ["content_hash_id", "client_hash_id",
             "log_impressions_90d", "avg_position_90d", "ctr_90d",
             "days_since_last_update", "content_age_days",
             "word_count", "has_word_count",
             "engagement_rate_90d", "sessions_90d",
             "is_declining_label"]
# Only show columns that exist
available_cols = [c for c in show_cols if c in feat_df.columns]
print(feat_df[available_cols].head(10).to_string(index=False))
print(f"\nShape: {feat_df.shape}")

## 5. The trap — deliberate leakage experiment

1. Add `trend_pct` (label-derived) as a "feature"
2. Fit DecisionTreeClassifier (depth=2) → precision@50 jumps toward 1.0
3. **Remove** leak column → retrain → precision drops to honest level
4. Keep honest number; note the lesson

In [ ]:
# Deliberate leakage experiment (uses feature frame from Section 4)
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np

# Recreate feat_df from Section 4 (re-run aggregation quickly)
feat_df = feat_df.copy()

# Synthetic label: low impressions in feature window (since trend_direction not in warehouse)
y = (feat_df["impressions_90d"] < feat_df["impressions_90d"].median()).astype(int)

clean_cols = ["log_impressions_90d", "avg_position_90d", "ctr_90d",
             "days_since_last_update", "content_age_days",
             "word_count", "has_word_count",
             "engagement_rate_90d", "sessions_90d"]

X_clean = feat_df[clean_cols].fillna(0)

# Client-grouped split
clients = feat_df["client_hash_id"].unique()
np.random.seed(42)
test_clients = set(np.random.choice(clients, size=max(1, len(clients)//5), replace=False))
test_mask = feat_df["client_hash_id"].isin(test_clients)

X_train, X_test = X_clean[~test_mask], X_clean[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

# --- Clean model ---
clf_clean = DecisionTreeClassifier(max_depth=2, random_state=42, min_samples_leaf=100)
clf_clean.fit(X_train, y_train)
proba_clean = clf_clean.predict_proba(X_test.fillna(0))[:, 1]
top50_clean = pd.Series(proba_clean, index=y_test.index).nlargest(min(50, len(y_test))).index
prec50_clean = y_test.loc[top50_clean].mean()

# --- LEAKY model: add label as "trend_pct" feature ---
X_leaky = X_clean.copy()
X_leaky["trend_pct_LEAK"] = y  # directly leak the label

X_train_leaky, X_test_leaky = X_leaky[~test_mask], X_leaky[test_mask]
clf_leaky = DecisionTreeClassifier(max_depth=2, random_state=42, min_samples_leaf=100)
clf_leaky.fit(X_train_leaky, y_train)
proba_leaky = clf_leaky.predict_proba(X_test_leaky.fillna(0))[:, 1]
top50_leaky = pd.Series(proba_leaky, index=y_test.index).nlargest(min(50, len(y_test))).index
prec50_leaky = y_test.loc[top50_leaky].mean()

print(f"Clean model precision@50: {prec50_clean:.3f}")
print(f"Leaky model precision@50:  {prec50_leaky:.3f}")
print(f"\nLeakage inflated precision by: {prec50_leaky - prec50_clean:.3f}")
print("\nLesson: Adding label-derived columns (trend_pct, trend_direction, target-window metrics)")
print("makes scores look perfect but the model learns the label, not the signal.")
print("Removed leak column — keeping honest precision.")

## 6. One named limitation

**Limitation: Unbalanced panel / per-client history depth.**

The warehouse spans 2025-01-27 → 2026-06-30, but `dim_clients.gsc_data_start` varies widely — some clients have 17 months of history, others only 3. A fixed 90-day calendar window (Dec 2025–Feb 2026) includes zero-filled GA4 rows for clients whose `ga4_data_start` is later (flagged `ga4_data_available = FALSE`). Those zeros mean "no tracking yet", not "no engagement". This limits the usable client set for a fixed calendar window and biases features toward clients with longer history.

A stronger contract would use **per-client windows** anchored to each client's `gsc_data_start` / `ga4_data_start` rather than one global calendar window.

In [ ]:
# Evidence: dim_clients gsc_data_start distribution
client_dist = dim_clients.groupby(["gsc_data_start", "ga4_data_start"]).size().reset_index(name="client_count")
client_dist = client_dist.sort_values("gsc_data_start")

print("Client history start dates (dim_clients):")
print(client_dist.to_string(index=False))

print(f"\nClients with GSC start before 2025-12-01 (enough for 90d feature window):")
early = client_dist[pd.to_datetime(client_dist['gsc_data_start']) < '2025-12-01']
print(f"  {early['client_count'].sum()} / {client_dist['client_count'].sum()} clients")

print(f"Clients with GA4 start before 2025-12-01:")
early_ga4 = client_dist[pd.to_datetime(client_dist['ga4_data_start']) < '2025-12-01']
print(f"  {early_ga4['client_count'].sum()} / {client_dist['client_count'].sum()} clients")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.